# 03 · Train the keyword spotter (DS-CNN)

* **Demo model** — ten English Speech Commands words, trains now from the
  downloaded data. Mapped to assistant actions in `assistant.py`:
  *stop* stop · *go* describe · *left/right* find menu · *yes* search · *no*
  cancel · *up/down* volume · *on/off* obstacle warnings.
* **Custom model** — your own Bangla + English recordings (notebook 02), with
  Speech Commands as *unknown*. This is the device's real vocabulary.

The split is **by speaker**: test accuracy is on voices the model never heard.

In [ ]:
import sys, pathlib
LAB = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(LAB))
from va_lab import config
config.ensure_dirs()
print("lab folder:", LAB)

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt
from va_lab import kws_data, kws_model
ds = kws_data.build_dataset("demo")
print(ds["classes"])
for split, counts in kws_data.summary(ds): print(split, counts)

In [ ]:
model, history = kws_model.train(ds, epochs=30, batch_size=256)
test = kws_model.evaluate(model, ds["testing"])
print(f"test accuracy (unseen speakers): {test['accuracy']:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot([h["train_acc"] for h in history], label="train"); ax[0].plot([h["val_acc"] for h in history], label="validation")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("accuracy"); ax[0].legend(); ax[0].grid(alpha=.3)
c = test["confusion"]; cn = c / c.sum(1, keepdims=True)
ax[1].imshow(cn, cmap="Blues"); ax[1].set_xticks(range(len(ds["classes"])), ds["classes"], rotation=60)
ax[1].set_yticks(range(len(ds["classes"])), ds["classes"]); ax[1].set_title("test confusion (rows = true)")
plt.tight_layout(); plt.show()
for i, name in enumerate(ds["classes"]): print(f"{name:10s} recall {cn[i, i]:.2f}")

In [ ]:
kws_model.save(model, ds["classes"], config.MODELS / "kws_demo.pt", history, vocab="demo")
kws_model.export_onnx(model, config.MODELS / "kws_demo.onnx")
print("saved", config.MODELS / "kws_demo.pt", "and .onnx;", sum(p.numel() for p in model.parameters()), "parameters")

## Try it: run the cell, wait for the tone, say a word

In [ ]:
from va_lab import audio_io
from va_lab.speech import Speaker, Utterance, ANSWER
import time
spot = kws_model.KeywordSpotter(config.MODELS / "kws_demo.pt")
Speaker(bangla=False).say(Utterance(ANSWER, [("tone", "listen")])); time.sleep(0.25)
label, conf, top = spot.classify(audio_io.record(config.KWS_LISTEN_S))
print(f"heard: {label} ({conf:.2f})   top3: {top}")

## Custom vocabulary (after recording in notebook 02)
Needs recordings from several speakers to be meaningful; with one speaker the
test split may be empty (the split is by speaker).

In [ ]:
try:
    dsc = kws_data.build_dataset("custom")
except RuntimeError as e:
    print(e)
else:
    for split, counts in kws_data.summary(dsc): print(split, counts)
    mc, hc = kws_model.train(dsc, epochs=40, batch_size=128)
    print("test accuracy:", kws_model.evaluate(mc, dsc["testing"])["accuracy"])
    kws_model.save(mc, dsc["classes"], config.MODELS / "kws_custom.pt", hc, vocab="custom")
    kws_model.export_onnx(mc, config.MODELS / "kws_custom.onnx")